In [1]:
# ============================================
# 1. Импорт библиотек
# ============================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


# ============================================
# 2. Загрузка данных
# ============================================


DATA_PATH = "data.xlsx"

# data = pd.read_excel(DATA_PATH, sheet_name="data")
# target = pd.read_excel(DATA_PATH, sheet_name="target")

data = pd.read_parquet("data.parquet")
target = pd.read_parquet("target.parquet")


print("Data shape:", data.shape)
print("Target shape:", target.shape)

display(data.head())
display(target.head())
display(data.dtypes)
display(target.dtypes)

Data shape: (4151096, 11)
Target shape: (157806, 2)


,h3_index,customer_id,datetime_id,count,sum,avg,min,max,std,count_distinct,mcc_code
0,8911aa4c62fffff,1,3,1,3346.65,3346.650,3346.65,3346.65,NaN,1,13
1,8911aa7b5b3ffff,4,3,1,450.00,450.000,450.00,450.00,NaN,1,8
2,8911aa63623ffff,5,3,10,11035.69,1103.569,59.00,3620.18,1190.530333,6,13
3,8911aa48577ffff,9,2,2,628.00,314.000,295.00,333.00,26.870058,2,5
4,8911aa78297ffff,11,2,1,4155.00,4155.000,4155.00,4155.00,NaN,1,10


,h3_index,customer_id
0,8911aa6ac3bffff,23172
1,8911aa7a857ffff,95640
2,8911aa70b97ffff,60350
3,8911aa70b97ffff,69521
4,891181b69abffff,29437


h3_index           object
customer_id         int64
datetime_id         int64
count               int64
sum               float64
avg               float64
min               float64
max               float64
std               float64
count_distinct      int64
mcc_code            int64
dtype: object

h3_index       object
customer_id     int64
dtype: object

In [2]:
# ============================================
# 3. Первичный осмотр структуры
# ============================================

print("DATA INFO")
data.info()

print("\nTARGET INFO")
target.info()

print("\nПропуски в data:")
display(data.isna().sum().sort_values(ascending=False))

print("\nПропуски в target:")
display(target.isna().sum().sort_values(ascending=False))

print("\nДубликаты в data:", data.duplicated().sum())
print("Дубликаты в target:", target.duplicated().sum())


DATA INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4151096 entries, 0 to 4151095
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   h3_index        object 
 1   customer_id     int64  
 2   datetime_id     int64  
 3   count           int64  
 4   sum             float64
 5   avg             float64
 6   min             float64
 7   max             float64
 8   std             float64
 9   count_distinct  int64  
 10  mcc_code        int64  
dtypes: float64(5), int64(5), object(1)
memory usage: 348.4+ MB

TARGET INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 157806 entries, 0 to 157805
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   h3_index     157806 non-null  object
 1   customer_id  157806 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 2.4+ MB

Пропуски в data:


std               2399551
h3_index                0
customer_id             0
datetime_id             0
count                   0
sum                     0
avg                     0
min                     0
max                     0
count_distinct          0
mcc_code                0
dtype: int64


Пропуски в target:


h3_index       0
customer_id    0
dtype: int64


Дубликаты в data: 501
Дубликаты в target: 0


In [3]:
# ============================================
# 4. Базовая статистика
# ============================================

numeric_cols = data.select_dtypes(include=[np.number]).columns

print("Числовые признаки:")
print(list(numeric_cols))

display(data[numeric_cols].describe().T)

Числовые признаки:
['customer_id', 'datetime_id', 'count', 'sum', 'avg', 'min', 'max', 'std', 'count_distinct', 'mcc_code']


,count,mean,std,min,25%,50%,75%,max
customer_id,4151096.0,46740.248076,27490.692232,1.00,22992.000000,46038.000000,69962.0000,9.845000e+04
datetime_id,4151096.0,2.100576,0.846003,0.00,2.000000,2.000000,3.0000,3.000000e+00
count,4151096.0,3.238023,6.292843,1.00,1.000000,1.000000,3.0000,4.150000e+02
sum,4151096.0,2633.186039,18580.131901,0.01,278.990000,755.000000,2200.0000,2.195500e+07
avg,4151096.0,1290.334265,10566.348232,0.01,183.333333,400.000000,980.0000,1.097750e+07
min,4151096.0,1087.152085,8965.371496,0.01,105.000000,269.970000,737.6525,9.001900e+06
max,4151096.0,1579.648991,14216.511618,0.01,210.000000,500.000000,1288.0000,1.540500e+07
std,1751545.0,591.171630,8948.862525,0.00,56.568542,174.513954,450.1701,6.261431e+06
count_distinct,4151096.0,1.736469,2.391839,0.00,1.000000,1.000000,2.0000,1.220000e+02
mcc_code,4151096.0,9.640391,4.075310,0.00,8.000000,9.000000,13.0000,2.300000e+01


In [4]:
# ============================================
# 5. Проверка ключевых сущностей
# ============================================

print("Количество уникальных H3-ячеек в data:", data["h3_index"].nunique())
print("Количество уникальных клиентов в data:", data["customer_id"].nunique())
print("Количество уникальных MCC в data:", data["mcc_code"].nunique())

print("\nКоличество уникальных H3-ячеек в target:", target["h3_index"].nunique())
print("Количество уникальных клиентов в target:", target["customer_id"].nunique())

Количество уникальных H3-ячеек в data: 8154
Количество уникальных клиентов в data: 69337
Количество уникальных MCC в data: 24

Количество уникальных H3-ячеек в target: 1657
Количество уникальных клиентов в target: 69337


In [5]:
# ============================================
# 6. Формирование целевой переменной по H3-ячейкам
# ============================================

# В target есть только положительные примеры.
# Если h3_index есть в target, считаем, что в ячейке была банкоматная активность.
# atm_activity_records - не нужна, так как в target у нас только уникальные пары
target_h3 = (
    target
    .groupby("h3_index")
    .agg(
        atm_active_customers=("customer_id", "nunique"),
        atm_activity_records=("customer_id", "count")
    )
    .reset_index()
)

target_h3["atm_activity"] = 1

display(target_h3)

print("Ячеек с банкоматной активностью:", target_h3.shape[0])

,h3_index,atm_active_customers,atm_activity_records,atm_activity
0,8911818610bffff,215,215,1
1,89118195133ffff,20,20,1
2,8911819513bffff,20,20,1
3,891181b2827ffff,58,58,1
4,891181b2957ffff,145,145,1
...,...,...,...,...
1652,8911aa7b65bffff,68,68,1
1653,8911aa7b663ffff,13,13,1
1654,8911aa7b67bffff,21,21,1
1655,8911aa7b687ffff,22,22,1


Ячеек с банкоматной активностью: 1657


In [6]:
# ============================================
# 7. Агрегация транзакций по H3-ячейкам
# ============================================

h3_features = (
    data
    .groupby("h3_index")
    .agg(
        total_records=("customer_id", "count"),
        unique_customers=("customer_id", "nunique"),
        total_count=("count", "sum"),
        total_sum=("sum", "sum"),
        avg_transaction=("avg", "mean"),
        min_transaction=("min", "min"),
        max_transaction=("max", "max"),
        avg_std=("std", "mean"),
        total_distinct_transactions=("count_distinct", "sum"),
        unique_mcc=("mcc_code", "nunique"),
        unique_time_buckets=("datetime_id", "nunique")
    )
    .reset_index()
)

# Дополнительные признаки

h3_features["sum_per_customer"] = (
    h3_features["total_sum"] / h3_features["unique_customers"].replace(0, np.nan)
)

h3_features["transactions_per_customer"] = (
    h3_features["total_count"] / h3_features["unique_customers"].replace(0, np.nan)
)

h3_features["records_per_customer"] = (
    h3_features["total_records"] / h3_features["unique_customers"].replace(0, np.nan)
)

display(h3_features.head())
print("Размер таблицы признаков:", h3_features.shape)

,h3_index,total_records,unique_customers,total_count,total_sum,avg_transaction,min_transaction,max_transaction,avg_std,total_distinct_transactions,unique_mcc,unique_time_buckets,sum_per_customer,transactions_per_customer,records_per_customer
0,89118180927ffff,14,10,22,18379.83,833.284667,84.98,2852.29,590.163753,18,2,3,1837.983000,2.200000,1.400000
1,89118180d27ffff,1,1,1,790.00,790.000000,790.00,790.00,NaN,1,1,1,790.000000,1.000000,1.000000
2,891181820abffff,311,248,499,834526.82,1731.767959,1.00,5552.80,479.022698,385,2,4,3365.027500,2.012097,1.254032
3,891181840a7ffff,3,3,3,12377.50,4125.833333,3277.50,4600.00,NaN,3,1,2,4125.833333,1.000000,1.000000
4,891181844c3ffff,1,1,1,246.00,246.000000,246.00,246.00,NaN,1,1,1,246.000000,1.000000,1.000000


Размер таблицы признаков: (8154, 15)


In [7]:
# ============================================
# 8. Объединение признаков с target
# ============================================

dataset = h3_features.merge(
    target_h3,
    on="h3_index",
    how="left"
)

dataset["atm_activity"] = dataset["atm_activity"].fillna(0).astype(int)
dataset["atm_active_customers"] = dataset["atm_active_customers"].fillna(0).astype(int)
dataset["atm_activity_records"] = dataset["atm_activity_records"].fillna(0).astype(int)

display(dataset.head())

print("Итоговый dataset:", dataset.shape)
print(dataset["atm_activity"].value_counts(normalize=True))

,h3_index,total_records,unique_customers,total_count,total_sum,avg_transaction,min_transaction,max_transaction,avg_std,total_distinct_transactions,unique_mcc,unique_time_buckets,sum_per_customer,transactions_per_customer,records_per_customer,atm_active_customers,atm_activity_records,atm_activity
0,89118180927ffff,14,10,22,18379.83,833.284667,84.98,2852.29,590.163753,18,2,3,1837.983000,2.200000,1.400000,0,0,0
1,89118180d27ffff,1,1,1,790.00,790.000000,790.00,790.00,NaN,1,1,1,790.000000,1.000000,1.000000,0,0,0
2,891181820abffff,311,248,499,834526.82,1731.767959,1.00,5552.80,479.022698,385,2,4,3365.027500,2.012097,1.254032,0,0,0
3,891181840a7ffff,3,3,3,12377.50,4125.833333,3277.50,4600.00,NaN,3,1,2,4125.833333,1.000000,1.000000,0,0,0
4,891181844c3ffff,1,1,1,246.00,246.000000,246.00,246.00,NaN,1,1,1,246.000000,1.000000,1.000000,0,0,0


Итоговый dataset: (8154, 18)
atm_activity
0    0.797155
1    0.202845
Name: proportion, dtype: float64


In [8]:
# ============================================
# 9. Сравнение зон с банкоматной активностью и без
# ============================================

comparison = (
    dataset
    .groupby("atm_activity")
    .agg(
        h3_count=("h3_index", "count"),
        avg_unique_customers=("unique_customers", "mean"),
        avg_total_count=("total_count", "mean"),
        avg_total_sum=("total_sum", "mean"),
        avg_transaction=("avg_transaction", "mean"),
        avg_unique_mcc=("unique_mcc", "mean"),
        avg_time_buckets=("unique_time_buckets", "mean"),
        avg_sum_per_customer=("sum_per_customer", "mean"),
        avg_transactions_per_customer=("transactions_per_customer", "mean")
    )
    .reset_index()
)

display(comparison)

,atm_activity,h3_count,avg_unique_customers,avg_total_count,avg_total_sum,avg_transaction,avg_unique_mcc,avg_time_buckets,avg_sum_per_customer,avg_transactions_per_customer
0,0,6500,128.496769,638.289538,4.818605e+05,2179.028309,3.723231,3.158923,4441.472186,4.180650
1,1,1654,610.387545,5618.176542,4.714943e+06,1369.607339,8.266626,3.637848,4628.029779,5.741085


In [9]:
# ============================================
# 10. Топ H3-ячеек по ключевым метрикам
# ============================================

top_by_sum = dataset.sort_values("total_sum", ascending=False).head(20)
top_by_customers = dataset.sort_values("unique_customers", ascending=False).head(20)
top_by_transactions = dataset.sort_values("total_count", ascending=False).head(20)

print("Топ-20 зон по сумме операций")
display(top_by_sum[[
    "h3_index", "total_sum", "unique_customers", "total_count", "atm_activity"
]])

print("Топ-20 зон по числу клиентов")
display(top_by_customers[[
    "h3_index", "unique_customers", "total_sum", "total_count", "atm_activity"
]])

print("Топ-20 зон по количеству операций")
display(top_by_transactions[[
    "h3_index", "total_count", "unique_customers", "total_sum", "atm_activity"
]])

Топ-20 зон по сумме операций


,h3_index,total_sum,unique_customers,total_count,atm_activity
7717,8911aa7abd3ffff,3.018260e+09,67487,2729155,1
3941,8911aa68e7bffff,8.856216e+08,51105,1034012,1
7309,8911aa7a107ffff,8.041047e+07,6190,49625,1
7598,8911aa7a967ffff,6.393434e+07,39550,874668,1
7313,8911aa7a117ffff,5.788189e+07,3815,137323,1
5401,8911aa7162fffff,5.719804e+07,4032,10502,0
5140,8911aa7103bffff,5.626898e+07,5549,20376,1
427,891181b6507ffff,4.680804e+07,5133,29984,1
3932,8911aa68e47ffff,4.292054e+07,12037,73218,0
3440,8911aa62627ffff,3.855564e+07,3416,16109,1


Топ-20 зон по числу клиентов


,h3_index,unique_customers,total_sum,total_count,atm_activity
7717,8911aa7abd3ffff,67487,3.018260e+09,2729155,1
3941,8911aa68e7bffff,51105,8.856216e+08,1034012,1
7598,8911aa7a967ffff,39550,6.393434e+07,874668,1
3932,8911aa68e47ffff,12037,4.292054e+07,73218,0
6923,8911aa792bbffff,11114,9.412435e+06,143469,0
3731,8911aa6360bffff,8671,1.775715e+07,35222,1
6235,8911aa78127ffff,6660,1.418625e+07,17855,1
7854,8911aa7aec7ffff,6527,3.739045e+07,26034,1
3936,8911aa68e63ffff,6287,2.215795e+07,29160,1
7309,8911aa7a107ffff,6190,8.041047e+07,49625,1


Топ-20 зон по количеству операций


,h3_index,total_count,unique_customers,total_sum,atm_activity
7717,8911aa7abd3ffff,2729155,67487,3.018260e+09,1
3941,8911aa68e7bffff,1034012,51105,8.856216e+08,1
7598,8911aa7a967ffff,874668,39550,6.393434e+07,1
6923,8911aa792bbffff,143469,11114,9.412435e+06,0
7313,8911aa7a117ffff,137323,3815,5.788189e+07,1
3932,8911aa68e47ffff,73218,12037,4.292054e+07,0
7077,8911aa7983bffff,61100,4197,1.026778e+07,0
7309,8911aa7a107ffff,49625,6190,8.041047e+07,1
941,8911aa01dc7ffff,43111,1917,3.326968e+07,1
7308,8911aa7a103ffff,38197,5471,3.120783e+07,1


In [10]:
pip install h3

Note: you may need to restart the kernel to use updated packages.


In [11]:
display(dataset.head())

,h3_index,total_records,unique_customers,total_count,total_sum,avg_transaction,min_transaction,max_transaction,avg_std,total_distinct_transactions,unique_mcc,unique_time_buckets,sum_per_customer,transactions_per_customer,records_per_customer,atm_active_customers,atm_activity_records,atm_activity
0,89118180927ffff,14,10,22,18379.83,833.284667,84.98,2852.29,590.163753,18,2,3,1837.983000,2.200000,1.400000,0,0,0
1,89118180d27ffff,1,1,1,790.00,790.000000,790.00,790.00,NaN,1,1,1,790.000000,1.000000,1.000000,0,0,0
2,891181820abffff,311,248,499,834526.82,1731.767959,1.00,5552.80,479.022698,385,2,4,3365.027500,2.012097,1.254032,0,0,0
3,891181840a7ffff,3,3,3,12377.50,4125.833333,3277.50,4600.00,NaN,3,1,2,4125.833333,1.000000,1.000000,0,0,0
4,891181844c3ffff,1,1,1,246.00,246.000000,246.00,246.00,NaN,1,1,1,246.000000,1.000000,1.000000,0,0,0


Продолжение

In [12]:
import h3
H3_RESOLUTION = 9

vtb_atms = pd.read_csv("vtb_atms.csv", sep=";")
offices = pd.read_csv("offices.csv", sep=";")
metro = pd.read_csv("metro.csv", sep=";")
competitor_atms = pd.read_csv("competitor_atms.csv", sep=";")
malls = pd.read_csv("malls.csv", sep=";")
universities = pd.read_csv("universities.csv", sep=";")

display(vtb_atms.head())
display(offices.head())
display(metro.head())
display(competitor_atms.head())
display(malls.head())
display(universities.head())

,atm_id,name,lat,lon
0,1,VTB ATM,55.778850,37.601728
1,2,VTB ATM,55.691654,37.534936
2,3,VTB ATM,55.727018,37.625356
3,4,VTB ATM,55.766132,37.637683
4,5,VTB ATM,55.722267,37.585671


,name,lat,lon
0,1,55.765363,37.627211
1,AZIMUT Сити Отель Комсити Москва,55.635920,37.430805
2,Action 44,55.769424,37.559348
3,Alstom Grid,55.790211,37.707712
4,Davis,55.804514,37.590304


,station,lat,lon
0,Авиамоторная,55.752535,37.720063
1,Автозаводская,55.707345,37.657768
2,Академическая,55.687808,37.573515
3,Александровский сад,55.752403,37.608683
4,Алексеевская,55.809089,37.639006


,atm_id,bank,lat,lon
0,1,Sber,55.759699,37.701532
1,2,Alfa-Bank,55.804281,37.523807
2,3,Alfa-Bank,55.743099,37.504192
3,4,Alfa-Bank,55.749713,37.704375
4,5,Alfa-Bank,55.704522,37.765949


,mall,lat,lon
0,1-й ТЦ,56.005784,37.206126
1,10-й торговый центр,55.984055,37.176077
2,1000 вещей,55.679844,37.666527
3,1000 мелочей,55.993273,37.213245
4,2-й торговый центр,56.002931,37.205715


,university,lat,lon
0,11 корпус РЭУ им. Г. В. Плеханова,55.712630,37.476814
1,12 корпус РЭУ им. Г. В. Плеханова,55.712094,37.477849
2,Moscow Business School,55.704561,37.569965
3,«Высшая школа экономики» АУЗ № 11,55.815746,37.524912
4,АНО ДПО Московская Высшая Школа Экономики,55.736726,37.659493


In [13]:
def add_h3_index(df, lat_col="lat", lon_col="lon", resolution=9):
    df = df.copy()
    
    df = df.dropna(subset=[lat_col, lon_col])
    
    df["h3_index"] = df.apply(
        lambda row: h3.latlng_to_cell(
            row[lat_col],
            row[lon_col],
            resolution
        ),
        axis=1
    )
    
    return df

In [14]:
vtb_atms = add_h3_index(vtb_atms, resolution=H3_RESOLUTION)
offices = add_h3_index(offices, resolution=H3_RESOLUTION)
metro = add_h3_index(metro, resolution=H3_RESOLUTION)
competitor_atms = add_h3_index(competitor_atms, resolution=H3_RESOLUTION)
malls = add_h3_index(malls, resolution=H3_RESOLUTION)
universities = add_h3_index(universities, resolution=H3_RESOLUTION)

display(vtb_atms.head())
display(offices.head())
display(metro.head())
display(competitor_atms.head())
display(malls.head())
display(universities.head())

,atm_id,name,lat,lon,h3_index
0,1,VTB ATM,55.778850,37.601728,8911aa7ad6fffff
1,2,VTB ATM,55.691654,37.534936,8911aa7b18bffff
2,3,VTB ATM,55.727018,37.625356,8911aa781abffff
3,4,VTB ATM,55.766132,37.637683,8911aa7ab33ffff
4,5,VTB ATM,55.722267,37.585671,8911aa7851bffff


,name,lat,lon,h3_index
0,1,55.765363,37.627211,8911aa7ab8fffff
1,AZIMUT Сити Отель Комсити Москва,55.635920,37.430805,8911aa4e8d7ffff
2,Action 44,55.769424,37.559348,8911aa7ac17ffff
3,Alstom Grid,55.790211,37.707712,8911aa633cfffff
4,Davis,55.804514,37.590304,8911aa71a83ffff


,station,lat,lon,h3_index
0,Авиамоторная,55.752535,37.720063,8911aa7896fffff
1,Автозаводская,55.707345,37.657768,8911aa7832fffff
2,Академическая,55.687808,37.573515,8911aa78613ffff
3,Александровский сад,55.752403,37.608683,8911aa7aaa3ffff
4,Алексеевская,55.809089,37.639006,8911aa634a3ffff


,atm_id,bank,lat,lon,h3_index
0,1,Sber,55.759699,37.701532,8911aa78917ffff
1,2,Alfa-Bank,55.804281,37.523807,8911aa711c7ffff
2,3,Alfa-Bank,55.743099,37.504192,8911aa7a547ffff
3,4,Alfa-Bank,55.749713,37.704375,8911aa78943ffff
4,5,Alfa-Bank,55.704522,37.765949,8911aa6a05bffff


,mall,lat,lon,h3_index
0,1-й ТЦ,56.005784,37.206126,8911aa00e5bffff
1,10-й торговый центр,55.984055,37.176077,8911aa00223ffff
2,1000 вещей,55.679844,37.666527,8911aa79c63ffff
3,1000 мелочей,55.993273,37.213245,8911aa01dc7ffff
4,2-й торговый центр,56.002931,37.205715,8911aa0032fffff


,university,lat,lon,h3_index
0,11 корпус РЭУ им. Г. В. Плеханова,55.712630,37.476814,8911aa45967ffff
1,12 корпус РЭУ им. Г. В. Плеханова,55.712094,37.477849,8911aa7a69bffff
2,Moscow Business School,55.704561,37.569965,8911aa7840bffff
3,«Высшая школа экономики» АУЗ № 11,55.815746,37.524912,8911aa711b3ffff
4,АНО ДПО Московская Высшая Школа Экономики,55.736726,37.659493,8911aa78c63ffff


In [15]:
vtb_atms_h3 = (
    vtb_atms
    .groupby("h3_index")
    .agg(
        vtb_atm_count=("h3_index", "count")
    )
    .reset_index()
)

offices_h3 = (
    offices
    .groupby("h3_index")
    .agg(
        vtb_office_count=("h3_index", "count")
    )
    .reset_index()
)

metro_h3 = (
    metro
    .groupby("h3_index")
    .agg(
        metro_count=("h3_index", "count")
    )
    .reset_index()
)

competitor_atms_h3 = (
    competitor_atms
    .groupby("h3_index")
    .agg(
        competitor_atm_count=("h3_index", "count")
    )
    .reset_index()
)

malls_h3 = (
    malls
    .groupby("h3_index")
    .agg(
        mall_count=("h3_index", "count")
    )
    .reset_index()
)

universities_h3 = (
    universities
    .groupby("h3_index")
    .agg(
        university_count=("h3_index", "count")
    )
    .reset_index()
)

In [16]:
external_h3_tables = [
    vtb_atms_h3,
    offices_h3,
    metro_h3,
    competitor_atms_h3,
    malls_h3,
    universities_h3
]

for table in external_h3_tables:
    dataset = dataset.merge(
        table,
        on="h3_index",
        how="left"
    )

In [17]:
external_count_cols = [
    "vtb_atm_count",
    "vtb_office_count",
    "metro_count",
    "competitor_atm_count",
    "mall_count",
    "university_count"
]

for col in external_count_cols:
    dataset[col] = dataset[col].fillna(0).astype(int)

In [18]:
display(dataset[[
    "h3_index",
    "total_sum",
    "unique_customers",
    "total_count",
    "atm_activity",
    "vtb_atm_count",
    "vtb_office_count",
    "metro_count",
    "competitor_atm_count",
    "mall_count",
    "university_count"
]].tail(20))

,h3_index,total_sum,unique_customers,total_count,atm_activity,vtb_atm_count,vtb_office_count,metro_count,competitor_atm_count,mall_count,university_count
8134,8911aa7b66bffff,80069.00,26,27,0,0,0,0,0,0,0
8135,8911aa7b66fffff,650555.57,253,1105,0,0,0,0,0,0,0
8136,8911aa7b673ffff,3630.00,7,26,0,0,0,0,0,0,0
8137,8911aa7b67bffff,57893.86,6,31,1,0,0,0,0,0,1
8138,8911aa7b687ffff,1114939.17,171,1835,1,0,0,0,0,0,0
8139,8911aa7b68bffff,698167.35,222,1078,0,0,0,0,0,0,0
8140,8911aa7b68fffff,933267.98,206,1648,1,0,0,0,0,0,0
8141,8911aa7b693ffff,611092.33,152,767,0,0,0,0,0,0,0
8142,8911aa7b697ffff,1042787.38,260,2147,0,0,0,0,0,0,0
8143,8911aa7b69bffff,234363.18,87,476,0,0,0,0,0,0,0


In [28]:
# dataset_with_external = dataset[
#     (dataset["vtb_atm_count"] > 0) |
#     (dataset["vtb_office_count"] > 0) |
#     (dataset["metro_count"] > 0) |
#     (dataset["competitor_atm_count"] > 0) |
#     (dataset["mall_count"] > 0) |
#     (dataset["university_count"] > 0)
# ]

# display(dataset_with_external[[
#     "h3_index",
#     "total_sum",
#     "unique_customers",
#     "total_count",
#     "atm_activity",
#     "vtb_atm_count",
#     "vtb_office_count",
#     "metro_count",
#     "competitor_atm_count",
#     "mall_count",
#     "university_count"
# ]].head(30))

# print(dataset_with_external.shape)

,h3_index,total_sum,unique_customers,total_count,atm_activity,vtb_atm_count,vtb_office_count,metro_count,competitor_atm_count,mall_count,university_count
25,8911818690bffff,21893.00,3,8,0,0,0,0,0,1,0
140,891181b2827ffff,1159351.54,194,1735,1,0,0,0,0,1,0
158,891181b2913ffff,33320.00,33,113,0,0,0,0,0,0,1
192,891181b2b83ffff,1676349.68,284,1516,1,0,0,0,0,1,0
201,891181b2bb3ffff,364478.52,119,446,1,0,0,0,0,1,0
250,891181b44b3ffff,162583.12,53,164,0,1,0,0,0,0,0
271,891181b45d7ffff,3041508.30,488,3802,1,0,0,0,1,1,0
283,891181b4c8fffff,5046804.31,595,5506,1,0,0,0,2,0,0
288,891181b4ca7ffff,622171.45,180,1055,1,0,0,1,0,0,0
291,891181b4cb3ffff,101497.97,53,106,1,0,0,0,1,0,0


(1522, 24)


In [20]:
display(dataset.head())

,h3_index,total_records,unique_customers,total_count,total_sum,avg_transaction,min_transaction,max_transaction,avg_std,total_distinct_transactions,unique_mcc,unique_time_buckets,sum_per_customer,transactions_per_customer,records_per_customer,atm_active_customers,atm_activity_records,atm_activity,vtb_atm_count,vtb_office_count,metro_count,competitor_atm_count,mall_count,university_count
0,89118180927ffff,14,10,22,18379.83,833.284667,84.98,2852.29,590.163753,18,2,3,1837.983000,2.200000,1.400000,0,0,0,0,0,0,0,0,0
1,89118180d27ffff,1,1,1,790.00,790.000000,790.00,790.00,NaN,1,1,1,790.000000,1.000000,1.000000,0,0,0,0,0,0,0,0,0
2,891181820abffff,311,248,499,834526.82,1731.767959,1.00,5552.80,479.022698,385,2,4,3365.027500,2.012097,1.254032,0,0,0,0,0,0,0,0,0
3,891181840a7ffff,3,3,3,12377.50,4125.833333,3277.50,4600.00,NaN,3,1,2,4125.833333,1.000000,1.000000,0,0,0,0,0,0,0,0,0
4,891181844c3ffff,1,1,1,246.00,246.000000,246.00,246.00,NaN,1,1,1,246.000000,1.000000,1.000000,0,0,0,0,0,0,0,0,0


In [21]:
dataset.to_csv("dataset_final.csv", index=False, encoding="utf-8-sig")

In [22]:
dataset.to_excel("dataset_final.xlsx", index=False)